# Vector Search & RAG

This notebook builds up from raw vector similarity to a full RAG pipeline backed by vector search, replacing the keyword-based retrieval from module 01.

In [1]:
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import numpy as np
import os
from minsearch import VectorSearch
from sqlitesearch import VectorSearchIndex
from openai import OpenAI
from dotenv import load_dotenv

from zoomcamp.rag_helper import RAGBase
from zoomcamp.ingest import load_faq_data, build_index

## 1. Semantic Similarity via Dot Product

`all-MiniLM-L6-v2` encodes text into 384-dimensional vectors. The dot product between two vectors measures semantic similarity — higher means more related. Two semantically similar sentences score high even with different wording; unrelated sentences score near zero.

In [2]:
q1 = "I just discovered the course, can I still join?"
q2 = "I just found out about the program, can I still enroll?"

In [3]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
v1 = model.encode(q1)

In [5]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [6]:
v1.dot(dv)

np.float32(0.39572877)

In [7]:
q3 = "How to install Docker on Windows?"
v3 = model.encode(q3)
v3.dot(dv)

np.float32(0.019730581)

## 2. Brute-Force Vector Search Over a Corpus

Encode all documents (question + answer concatenated) into a matrix `X` of shape `(n_docs, 384)`. A single matrix multiply `X.dot(v_query)` scores every document at once. `np.argsort` retrieves top-k results — no index library, just NumPy.

In [8]:
documents = load_faq_data()

texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [9]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [10]:
X = np.array(vectors)
X.shape

(1350, 384)

In [11]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)
scores = X.dot(v_query)

In [12]:
idx = np.argmax(scores)
print(idx, scores[idx])
print(documents[idx])

2 0.762941
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}


In [13]:
top5 = np.argsort(scores)[-5:]

for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.56009996
{'id': '068529125b', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I follow the course after it finishes?', 'answer': 'Yes, we will keep all the materials available, so you can follow the course at your own pace after it finishes.\n\nYou can also continue reviewing the homeworks and prepare for the next cohort. You can also start working on your final capstone project.'}

0.6536311
{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

0.7192132
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The course has already started. Can I still join it?', 'answer': 'Yes, you can. Even though you missed the start date

## 3. `VectorSearch` from `minsearch`

Wraps the brute-force matrix search in a clean API. `.fit(X, documents)` stores the matrix and metadata. `.search()` handles encoding scores and returning ranked documents. The `filter_dict` parameter narrows candidates by keyword field (e.g. `course`) before vector ranking.

In [14]:
vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [15]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [16]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [17]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

## 4. RAG Pipeline — Keyword Retrieval (Baseline)

`RAGBase` from the project's `zoomcamp` package wires a search index to an LLM. Here it uses the keyword-based `minsearch` index from module 01 as a baseline before we swap in vector retrieval.

In [18]:
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

openai_client = OpenAI(api_key=api_key)

In [19]:
documents = load_faq_data()
index = build_index(documents)

In [20]:
assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    model="gpt-5.4-mini"
)

In [21]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

## 5. `RAGVector` — RAG with Vector Retrieval

Subclasses `RAGBase` and overrides only the `.search()` method. The query is encoded at runtime, then passed to the `VectorSearch` index with a course filter. Everything else (prompt building, LLM call, response parsing) is inherited unchanged.

In [22]:

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [23]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
    model="gpt-5.4-mini"
)

In [24]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still join. If you want to receive a certificate, make sure to submit your project while submissions are still open.'

## 6. Persistent Vector Index with `sqlitesearch`

`VectorSearchIndex` persists embeddings to a SQLite database (`mode='ivf'` uses an inverted file index for approximate nearest-neighbor search). Same `.search()` and `filter_dict` API as `minsearch`, but survives process restarts. Call `.close()` to flush and release the file handle.

In [25]:
vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='db/faq_vectors2.db'
)

In [26]:
query = 'I just discovered the course. Can I still join it?'
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [27]:
results = vs_index.search(
    query_vector,
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

results


[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co

In [28]:
vs_index.close()

In [29]:
vector_assistant.rag('whats queen gambit?')

"I don't know."